[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C36_GPU_Kernels_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **模拟 GPU 编程模型**，再与朴素参考实现 **对拍**。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会「**算力 ≫ 带宽**」为什么让多数算子访存受限；③ 立下全课的纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画 roofline）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 真实 GPU 的「算力 ≫ 带宽」缺口

用几款真实 GPU 的公开规格，算一个关键比值：**每从 HBM 搬运 1 字节（FP16，2 字节/数），硬件有空做多少次浮点运算？**

这个比值就是 roofline 的 **脊点（ridge point）算术强度**。算子的算术强度低于它 → 访存受限；高于它 → 算力受限。

In [ ]:
# 真实 GPU 公开规格（约数；FP16/BF16 张量核心峰值算力 与 HBM 带宽）
GPUS = {
    'V100  (2017)': dict(tflops=125.0,  tbps=0.90),
    'A100  (2020)': dict(tflops=312.0,  tbps=2.039),
    'H100  (2022)': dict(tflops=989.0,  tbps=3.35),
}
print(f"{'GPU':14s} {'算力(TFLOP/s)':>14s} {'带宽(TB/s)':>12s} {'脊点 FLOP/byte':>16s}")
for name, s in GPUS.items():
    flops = s['tflops'] * 1e12
    bytes_per_s = s['tbps'] * 1e12
    ridge = flops / bytes_per_s          # FLOP per byte at the ridge point
    print(f'{name:14s} {s["tflops"]:>14.0f} {s["tbps"]:>12.3f} {ridge:>16.1f}')
print('\n解读：脊点高达几十~几百 FLOP/byte。一个算子要不被带宽卡住，')
print('每读 1 字节就得做这么多次运算——逐元素算子(每元素~1次)远远不够，注定访存受限。')

**关键结论**：脊点算术强度 = 峰值算力 ÷ 带宽，通常高达几十到几百 FLOP/byte。

逐元素算子、softmax、LayerNorm 的算术强度只有 ~O(1) FLOP/byte，**远在脊点左侧 → 访存受限**。这就是为什么本课把大量精力花在「少搬数据」（合并访问、分块、融合）而非「少算」上。

## 3 · 算一个具体算子的算术强度

以逐元素 `y = a * x + b`（每个元素 2 FLOPs：一乘一加）为例。读 `x`、写 `y`，FP32 各 4 字节。

In [ ]:
def arithmetic_intensity(flops, bytes_moved):
    return flops / bytes_moved          # FLOP per byte

n = 1_000_000
flops = 2 * n                            # 每元素 1 乘 1 加
bytes_moved = 4 * n + 4 * n              # 读 x(4B) + 写 y(4B), 忽略标量 a,b
ai = arithmetic_intensity(flops, bytes_moved)
print(f'逐元素 a*x+b 的算术强度 = {ai:.3f} FLOP/byte')
assert abs(ai - 0.25) < 1e-9            # 2n / 8n = 0.25
print('远小于任何 GPU 的脊点(几十~几百) -> 铁定访存受限 ✅')

## 4 · 立纪律：对拍（differential testing）

本课每个「内核」都要和一个**朴素参考实现**比对，标准是 `np.allclose(kernel, ref, atol=1e-10)`。

先把这个工作流跑通：写一个朴素 softmax 当参考，再写一个「数值稳定」版当被测内核，对拍它们**数学上等价**。

In [ ]:
def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def softmax_stable(x):
    # 减去每行最大值再 exp：数学等价，但不会溢出
    m = x.max(axis=-1, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=-1, keepdims=True)

rng = np.random.default_rng(0)
x = rng.standard_normal((4, 8))
assert np.allclose(softmax_naive(x), softmax_stable(x), atol=1e-12)
print('对拍通过：稳定版与朴素版数值一致 ✅')

# 现在展示朴素版的危险：大 logit 直接溢出
big = np.array([[1000.0, 1001.0, 1002.0]])
out_naive = softmax_naive(big)
out_stable = softmax_stable(big)
print('朴素版（溢出）   :', out_naive)        # 含 nan
print('稳定版（正确）   :', np.round(out_stable, 4))
assert np.isnan(out_naive).any(), '朴素版应当溢出为 nan'
assert not np.isnan(out_stable).any()
print('\n教训：数学等价的两种写法，数值行为可能天差地别 —— 这是模块 04/05 的伏笔。')

## 5 · 内存层级：越靠上越快越小

GPU 的内存是一个金字塔。下面用**近似**的相对延迟/带宽数量级（不同架构有别，量级正确即可）建一张表，建立「为什么要把数据搬到上层并复用」的直觉——这是模块 02/03 的主线。

In [ ]:
# 相对量级（以 global/HBM 为基准；真实值随架构变化，这里只看数量级）
HIER = [
    # (层级, 作用域, 相对延迟(周期), 相对带宽(x HBM))
    ('register',      'per-thread', 1,    400.0),
    ('shared/L1',     'per-block',  30,   30.0),
    ('L2 cache',      'all SMs',    200,  5.0),
    ('global/HBM',    'all',        500,  1.0),
]
print(f"{'层级':<14}{'作用域':<12}{'延迟(周期)':>10}{'带宽(xHBM)':>12}")
for name, scope, lat, bw in HIER:
    print(f'{name:<14}{scope:<12}{lat:>10}{bw:>12.0f}')
# 单调性检查：越往下延迟越大、带宽越小
lats = [h[2] for h in HIER]; bws = [h[3] for h in HIER]
assert lats == sorted(lats), '延迟应随层级递增'
assert bws == sorted(bws, reverse=True), '带宽应随层级递减'
print('\n✅ 金字塔单调性成立：register 最快最窄(私有)，HBM 最慢但最大(共享)。')
print('性能工程主线 = 把数据从 HBM 搬到 shared/register 并反复复用，少碰 HBM。')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成一个小函数，后面每个模块都用它判定「我的内核 == 朴素参考」。它就是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_allclose(name, got, ref, atol=1e-10):
    '''对拍：被测内核结果 vs 朴素参考。返回是否一致并打印。'''
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(np.asarray(got) - np.asarray(ref)))) if np.size(got) else 0.0
    print(f'[{name:<22}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：一个「分块求和」内核 对拍 np.sum
def blocked_sum(x, block=16):
    total = 0.0
    for i in range(0, len(x), block):
        total += x[i:i+block].sum()      # 每块先局部求和（模拟 shared 内规约）
    return total

x = rng.standard_normal(1000)
check_allclose('blocked_sum vs np.sum', blocked_sum(x), x.sum())
print('\n这就是全课的工作流：写内核 -> 对拍朴素参考 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个分块/规约/在线-softmax 内核，都会用 `np.allclose` 对拍朴素参考；结构正确则数值一致，数值一致则逻辑可迁移到 Triton/CUDA。

**接下来六个模块**：01 执行模型 → 02 内存与合并访问 → 03 分块矩阵乘 → 04 规约与 online softmax → 05 FlashAttention。每一步都建立在前一步之上。

下一站：**模块 01 · GPU 执行模型**。